In [4]:
#installing
!pip install simpy

#importing
import random
import simpy

#setting random seed for reproducibility
seed = 123

#total boats that plan to race
total_boats = 204
#boat interarrival distribution parameters. Exponential with a mean of 2 (1/2 = .5)
lambda_arrive = .5 
#stake boat lock on distribution parameters for experienced rowers
lower_lock_exp = 1
upper_lock_exp = 3
#stake boat lock on distribution parameters for novice rowers
lower_lock_nov = 1
upper_lock_nov = 5
#race distribution parameters
mu_race = 7.5
sigma_race = 1.25
#number of novice races in the regatta
nov_races = 12

#defining processes
class RowingRegatta:
    def __init__(self, env, boats_per_race):
        #initialize environment
        self.env = env
        #intialize number of boats per race
        self.boats_per_race = boats_per_race
        #boat count starts at 0
        self.boat_count = 0
        #creating a list of true/false to determine if the race is experienced or novice
        self.is_exp = [True] * nov_races + [False] * (total_boats // boats_per_race - nov_races)
        random.shuffle(self.is_exp)
        #start the run process everytime an instance is created
        self.action = env.process(self.run_regatta())
        #will record start times of races to compute the average race center
        self.race_start_times = []
        #will record the race type of experienced or novice
        self.race_types = []

    def boat(self, bow_num, start_flag):
        #boat arrives event
        arrival_time = random.expovariate(lambda_arrive)
        yield self.env.timeout(arrival_time)
        print(f'Bow Number {bow_num} arrived at time {self.env.now}')
        #is boat experienced or novice?
        is_exp = self.is_exp[(bow_num - 1) % self.boats_per_race]
        #boat gets locked on to stake boat event
        if is_exp:
            lockon_time = random.uniform(lower_lock_exp, upper_lock_exp)
        else:
            lockon_time = random.uniform(lower_lock_nov, upper_lock_nov)
        yield self.env.timeout(lockon_time)
        print(f'Bow Number {bow_num} locked on to stake boat at time {self.env.now}')
        #all boats waits for start flag event
        yield start_flag
        #boat races event
        race_time = random.gauss(mu_race, sigma_race)
        yield self.env.timeout(race_time)
        
    def run_regatta(self):
        #checking that we still have boats that haven't raced yet
        while self.boat_count < total_boats:
            #recording boats in a race
            boats = []
            #common event for all boats to start race at same time
            start_flag = self.env.event()  
            #getting the maximum number of boats in per race
            for _ in range(self.boats_per_race):
                if self.boat_count >= total_boats:
                    break
                self.boat_count += 1
                boats.append(env.process(self.boat(self.boat_count, start_flag)))
            #recording race type
            race_type = "Experienced" if any(self.is_exp[(self.boat_count - 1) % self.boats_per_race:
                                                         (self.boat_count - 1) % self.boats_per_race + 6]) else "Novice"
            self.race_types.append(race_type)
            #recording when race starts
            race_start_time = self.env.now
            self.race_start_times.append(race_start_time)
            #pauses the process for 0 minutes so the start race event can be triggered
            yield self.env.timeout(0)
            #start race event triggered
            start_flag.succeed()
            #waiting for all boats to finish race event
            yield self.env.all_of(boats)
            print(f'{race_type} Race finished at time {self.env.now}')
        
        #calculating average race center
        if len(self.race_start_times) > 1:
            race_centers = [
                self.race_start_times[i] - self.race_start_times[i - 1]
                for i in range(1, len(self.race_start_times))]
            average_center = sum(race_centers) / len(race_centers)
            print(f'Average race center: {average_center:.2f} minutes')        
        
#set up and run sim
print('Rowing Regatta')
random.seed(seed)
#create environment and setup
env = simpy.Environment()
regatta = RowingRegatta(env, boats_per_race=6)
#run sim
env.run()

Rowing Regatta
Bow Number 2 arrived at time 0.26992128238903124
Bow Number 4 arrived at time 0.6900617684465471
Bow Number 5 arrived at time 1.1264762372523467
Bow Number 6 arrived at time 1.296054671364965
Bow Number 4 locked on to stake boat at time 1.8370351330618309
Bow Number 1 arrived at time 2.2000555867542753
Bow Number 2 locked on to stake boat at time 3.147091989311969
Bow Number 5 locked on to stake boat at time 3.3468647493708694
Bow Number 6 locked on to stake boat at time 4.38273567378803
Bow Number 1 locked on to stake boat at time 4.477568711206433
Bow Number 3 arrived at time 4.7418987142811515
Bow Number 3 locked on to stake boat at time 8.326516999959543
Experienced Race finished at time 16.03463559144685
Bow Number 7 arrived at time 17.174324805143236
Bow Number 12 arrived at time 17.50511306861511
Bow Number 8 arrived at time 17.581266255373652
Bow Number 10 arrived at time 17.849815780236547
Bow Number 7 locked on to stake boat at time 19.013317302088524
Bow Numbe